## Step 1: Install Dependencies and Import Libraries

In [1]:
import os
import sys
from pathlib import Path

# Prefer Java 17/21 for Spark on Windows (Java 23 causes Hadoop UGI getSubject() failures)
java_candidates = [
    r"C:\Program Files\Microsoft\jdk-17.0.18.8-hotspot",
    r"C:\Program Files\Microsoft\jdk-17",
    r"C:\Program Files\Java\jdk-21",
    r"C:\Program Files\Java\jdk-17",
]

current_java = os.environ.get("JAVA_HOME", "")
if (not current_java) or ("jdk-23" in current_java.lower()):
    for candidate in java_candidates:
        if Path(candidate).exists():
            os.environ["JAVA_HOME"] = candidate
            break

# Set local Hadoop home (contains winutils.exe) for Spark writes on Windows
_repo = Path(".").resolve()
_hadoop_home = _repo / "tools" / "hadoop"
os.environ["HADOOP_HOME"] = str(_hadoop_home)
os.environ["hadoop.home.dir"] = str(_hadoop_home)
os.environ["PATH"] = f"{_hadoop_home / 'bin'};{os.environ.get('PATH', '')}"

# Fix for Windows hostnames with underscores — Spark RPC rejects them
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

# Install PySpark (Colab only - skip if running locally)
# !pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time
import json
import gc

print("✅ Libraries imported successfully")
print(f"   JAVA_HOME       : {os.environ.get('JAVA_HOME')}")
print(f"   HADOOP_HOME     : {os.environ.get('HADOOP_HOME')}")
print(f"   SPARK_LOCAL_IP  : {os.environ.get('SPARK_LOCAL_IP')}")
print(f"   Python          : {sys.version.split()[0]}")

✅ Libraries imported successfully
   JAVA_HOME       : C:\Program Files\Microsoft\jdk-17.0.18.8-hotspot
   HADOOP_HOME     : D:\Coding\network-intrusion-detection\tools\hadoop
   SPARK_LOCAL_IP  : 127.0.0.1
   Python          : 3.11.9


## Step 2: Configure Paths

In [2]:
# Configure paths based on environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/NetworkIDS"
    TRAIN_PATH = f"{BASE_DIR}/data/training/UNSW_NB15_training-set.csv"
    TEST_PATH = f"{BASE_DIR}/data/testing/UNSW_NB15_testing-set.csv"
    MODEL_DIR = f"{BASE_DIR}/models"
    IS_COLAB = True
    print(f"✅ Google Drive mounted successfully!")
except:
    import pathlib
    BASE_DIR = str(pathlib.Path("__file__" if "__file__" in dir() else ".").resolve().parent)
    # Use paths relative to repo root
    _repo = pathlib.Path(".").resolve()
    TRAIN_PATH = str(_repo / "data" / "UNSW-NB15" / "UNSW_NB15_training-set.csv")
    TEST_PATH  = str(_repo / "data" / "UNSW-NB15" / "UNSW_NB15_testing-set.csv")
    MODEL_DIR  = str(_repo / "models")
    IS_COLAB = False
    print(f"✅ Running locally")
    print(f"   Repo root : {_repo}")

os.makedirs(MODEL_DIR, exist_ok=True)

print(f"📂 Training data: {TRAIN_PATH}")
print(f"📂 Testing data: {TEST_PATH}")
print(f"📂 Model directory: {MODEL_DIR}")

✅ Running locally
   Repo root : D:\Coding\network-intrusion-detection
📂 Training data: D:\Coding\network-intrusion-detection\data\UNSW-NB15\UNSW_NB15_training-set.csv
📂 Testing data: D:\Coding\network-intrusion-detection\data\UNSW-NB15\UNSW_NB15_testing-set.csv
📂 Model directory: D:\Coding\network-intrusion-detection\models


## Step 3: Create Spark Session

In [3]:
# Force garbage collection before creating session
gc.collect()

# Create Spark session optimized for ML training
spark = SparkSession.builder \
    .appName("UNSW-NB15-ModelTraining") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "600") \
    .config("spark.memory.fraction", "0.6") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark session created")
print(f"📊 Spark version: {spark.version}")

✅ Spark session created
📊 Spark version: 4.1.1


In [4]:
# Quick Windows Hadoop write smoke test (same path used by model/scaler saves)
smoke_path = str(Path(MODEL_DIR) / "_smoke_write_test")
spark.range(5).write.mode("overwrite").parquet(smoke_path)
print(f"✅ Spark write smoke test passed: {smoke_path}")

✅ Spark write smoke test passed: D:\Coding\network-intrusion-detection\models\_smoke_write_test


## Step 4: Load UNSW-NB15 Dataset

In [5]:
# Load training and testing data
print("Loading UNSW-NB15 dataset...")
start_time = time.time()

train_df = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True)
test_df = spark.read.csv(TEST_PATH, header=True, inferSchema=True)

print(f"✅ Dataset loaded in {time.time() - start_time:.2f} seconds")
print(f"📊 Training records: {train_df.count():,}")
print(f"📊 Testing records: {test_df.count():,}")
print(f"📊 Total features: {len(train_df.columns)}")

Loading UNSW-NB15 dataset...
✅ Dataset loaded in 4.59 seconds
📊 Training records: 82,332
📊 Testing records: 175,341
📊 Total features: 45


In [6]:
# Show schema
print("Dataset Schema (first 15 columns):")
for i, field in enumerate(train_df.schema.fields[:15]):
    print(f"  {field.name}: {field.dataType}")

Dataset Schema (first 15 columns):
  id: IntegerType()
  dur: DoubleType()
  proto: StringType()
  service: StringType()
  state: StringType()
  spkts: IntegerType()
  dpkts: IntegerType()
  sbytes: IntegerType()
  dbytes: IntegerType()
  rate: DoubleType()
  sttl: IntegerType()
  dttl: IntegerType()
  sload: DoubleType()
  dload: DoubleType()
  sloss: IntegerType()


In [7]:
# Check label distribution
print("\n📊 Attack Category Distribution (Training):")
train_df.groupBy("attack_cat").count().orderBy(F.desc("count")).show()

print("\n📊 Binary Label Distribution (Training):")
train_df.groupBy("label").count().show()


📊 Attack Category Distribution (Training):
+--------------+-----+
|    attack_cat|count|
+--------------+-----+
|        Normal|37000|
|       Generic|18871|
|      Exploits|11132|
|       Fuzzers| 6062|
|           DoS| 4089|
|Reconnaissance| 3496|
|      Analysis|  677|
|      Backdoor|  583|
|     Shellcode|  378|
|         Worms|   44|
+--------------+-----+


📊 Binary Label Distribution (Training):
+-----+-----+
|label|count|
+-----+-----+
|    0|37000|
|    1|45332|
+-----+-----+



## Step 5: Data Preprocessing

In [8]:
# Define feature columns (exclude id, label columns, and categorical)
NUMERIC_FEATURES = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sttl", "dttl",
    "sload", "dload", "sloss", "dloss", "sinpkt", "dinpkt", "sjit", "djit",
    "swin", "stcpb", "dtcpb", "dwin", "tcprtt", "synack", "ackdat",
    "smean", "dmean", "trans_depth", "response_body_len", "ct_srv_src",
    "ct_state_ttl", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd",
    "ct_src_ltm", "ct_srv_dst", "is_sm_ips_ports"
]

# Attack category mapping
ATTACK_MAPPING = {
    "Normal": 0,
    "Fuzzers": 1,
    "Analysis": 2,
    "Backdoors": 3,
    "Backdoor": 3,
    "DoS": 4,
    "Exploits": 5,
    "Generic": 6,
    "Reconnaissance": 7,
    "Shellcode": 8,
    "Worms": 9
}

print(f"📊 Using {len(NUMERIC_FEATURES)} numeric features")
print(f"📊 {len(ATTACK_MAPPING)} attack categories")

📊 Using 39 numeric features
📊 11 attack categories


In [9]:
def preprocess_unsw_data(df):
    """Preprocess UNSW-NB15 data: clean, handle nulls, create labels"""
    
    # Clean attack_cat column
    df = df.withColumn("attack_cat", F.trim(F.col("attack_cat")))
    df = df.withColumn("attack_cat", 
        F.when(F.col("attack_cat").isNull() | (F.col("attack_cat") == ""), "Normal")
        .otherwise(F.col("attack_cat")))
    
    # Create binary label (0 = Normal, 1 = Attack)
    df = df.withColumn("binary_label",
        F.when(F.col("attack_cat") == "Normal", 0.0).otherwise(1.0))
    
    # Create multiclass label from mapping
    label_expr = F.lit(0)  # Default to Normal
    for attack, label in ATTACK_MAPPING.items():
        label_expr = F.when(F.col("attack_cat") == attack, label).otherwise(label_expr)
    df = df.withColumn("multiclass_label", label_expr.cast(DoubleType()))
    
    # Handle null/infinite values in numeric features
    for col in NUMERIC_FEATURES:
        if col in df.columns:
            df = df.withColumn(col,
                F.when(F.col(col).isNull(), 0.0)
                .when(F.col(col) == float("inf"), 0.0)
                .when(F.col(col) == float("-inf"), 0.0)
                .otherwise(F.col(col).cast(DoubleType())))
    
    return df

print("Preprocessing training data...")
train_df = preprocess_unsw_data(train_df)

print("Preprocessing testing data...")
test_df = preprocess_unsw_data(test_df)

print("✅ Data preprocessing complete")

Preprocessing training data...
Preprocessing testing data...
✅ Data preprocessing complete


In [10]:
# Verify label distributions after preprocessing
print("\n📊 Binary Label Distribution (Training):")
train_df.groupBy("binary_label").count().show()

print("\n📊 Multiclass Label Distribution (Training):")
train_df.groupBy("multiclass_label", "attack_cat").count().orderBy("multiclass_label").show()


📊 Binary Label Distribution (Training):
+------------+-----+
|binary_label|count|
+------------+-----+
|         0.0|37000|
|         1.0|45332|
+------------+-----+


📊 Multiclass Label Distribution (Training):
+----------------+--------------+-----+
|multiclass_label|    attack_cat|count|
+----------------+--------------+-----+
|             0.0|        Normal|37000|
|             1.0|       Fuzzers| 6062|
|             2.0|      Analysis|  677|
|             3.0|      Backdoor|  583|
|             4.0|           DoS| 4089|
|             5.0|      Exploits|11132|
|             6.0|       Generic|18871|
|             7.0|Reconnaissance| 3496|
|             8.0|     Shellcode|  378|
|             9.0|         Worms|   44|
+----------------+--------------+-----+



## Step 6: Feature Engineering

In [11]:
# Get available features (some may be missing)
available_features = [col for col in NUMERIC_FEATURES if col in train_df.columns]
print(f"📊 Available features: {len(available_features)}/{len(NUMERIC_FEATURES)}")
print(f"Features: {available_features}")

📊 Available features: 39/39
Features: ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [12]:
# Assemble features into vector
print("\n🔧 Assembling features...")
assembler = VectorAssembler(
    inputCols=available_features,
    outputCol="features_raw",
    handleInvalid="skip"
)

train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

print("✅ Feature vectors assembled")


🔧 Assembling features...
✅ Feature vectors assembled


In [13]:
# Scale features using StandardScaler
print("\n🔧 Scaling features...")
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)

# Fit scaler on training data only
scaler_model = scaler.fit(train_df)

# Transform both train and test
train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

print("✅ Features scaled")


🔧 Scaling features...
✅ Features scaled


In [14]:
# Save the scaler model
scaler_path = f"{MODEL_DIR}/unsw_scaler"
scaler_model.write().overwrite().save(scaler_path)
print(f"✅ Scaler saved to: {scaler_path}")

✅ Scaler saved to: D:\Coding\network-intrusion-detection\models/unsw_scaler


## Step 7: Calculate Class Weights for Imbalanced Data

In [15]:
# Calculate class weights for binary classification
print("📊 Calculating class weights for binary classification...")
binary_counts = train_df.groupBy("binary_label").count().collect()
total_binary = sum(row['count'] for row in binary_counts)

binary_weights = {}
for row in binary_counts:
    label = row['binary_label']
    count = row['count']
    # Inverse frequency weighting with sqrt smoothing
    weight = (total_binary / (2 * count)) ** 0.5
    binary_weights[label] = weight
    print(f"  Class {int(label)}: {count:,} samples -> weight {weight:.4f}")

# Apply binary weights
binary_weight_expr = F.lit(1.0)
for label, weight in binary_weights.items():
    binary_weight_expr = F.when(F.col("binary_label") == label, weight).otherwise(binary_weight_expr)

train_df = train_df.withColumn("binary_weight", binary_weight_expr)

📊 Calculating class weights for binary classification...
  Class 0: 37,000 samples -> weight 1.0548
  Class 1: 45,332 samples -> weight 0.9529


In [16]:
# Calculate class weights for multiclass classification
print("\n📊 Calculating class weights for multiclass classification...")
multi_counts = train_df.groupBy("multiclass_label").count().collect()
total_multi = sum(row['count'] for row in multi_counts)
num_classes = len(multi_counts)

multi_weights = {}
for row in multi_counts:
    label = row['multiclass_label']
    count = row['count']
    # Inverse frequency weighting with sqrt smoothing
    weight = (total_multi / (num_classes * count)) ** 0.5
    multi_weights[label] = weight
    attack_name = [k for k, v in ATTACK_MAPPING.items() if v == int(label)][0] if label in range(10) else "Unknown"
    print(f"  Class {int(label)} ({attack_name}): {count:,} samples -> weight {weight:.4f}")

# Apply multiclass weights
multi_weight_expr = F.lit(1.0)
for label, weight in multi_weights.items():
    multi_weight_expr = F.when(F.col("multiclass_label") == label, weight).otherwise(multi_weight_expr)

train_df = train_df.withColumn("multiclass_weight", multi_weight_expr)


📊 Calculating class weights for multiclass classification...
  Class 8 (Shellcode): 378 samples -> weight 4.6670
  Class 0 (Normal): 37,000 samples -> weight 0.4717
  Class 7 (Reconnaissance): 3,496 samples -> weight 1.5346
  Class 1 (Fuzzers): 6,062 samples -> weight 1.1654
  Class 6 (Generic): 18,871 samples -> weight 0.6605
  Class 5 (Exploits): 11,132 samples -> weight 0.8600
  Class 4 (DoS): 4,089 samples -> weight 1.4190
  Class 3 (Backdoors): 583 samples -> weight 3.7579
  Class 9 (Worms): 44 samples -> weight 13.6791
  Class 2 (Analysis): 677 samples -> weight 3.4873


In [17]:
# Cache prepared data
print("\n🗃️ Caching prepared data...")
train_df = train_df.select(
    "features_scaled", "binary_label", "multiclass_label", 
    "binary_weight", "multiclass_weight", "attack_cat"
).cache()

test_df = test_df.select(
    "features_scaled", "binary_label", "multiclass_label", "attack_cat"
).cache()

# Materialize cache
train_count = train_df.count()
test_count = test_df.count()

print(f"✅ Training set: {train_count:,} records")
print(f"✅ Test set: {test_count:,} records")
print("✅ Data ready for training!")


🗃️ Caching prepared data...
✅ Training set: 82,332 records
✅ Test set: 175,341 records
✅ Data ready for training!


## Step 8: Train Binary Classification Models

### 8.1 Random Forest - Binary Classification

In [18]:
# Random Forest for Binary Classification
print("="*60)
print("Training Random Forest - Binary Classification")
print("="*60)

start_time = time.time()

rf_binary = RandomForestClassifier(
    featuresCol='features_scaled',
    labelCol='binary_label',
    weightCol='binary_weight',
    numTrees=200,
    maxDepth=18,
    maxBins=64,
    minInstancesPerNode=3,
    featureSubsetStrategy='sqrt',
    seed=42
)

print("🚀 Training model...")
rf_binary_model = rf_binary.fit(train_df)
elapsed = time.time() - start_time
print(f"✅ Training completed in {elapsed/60:.2f} minutes")

Training Random Forest - Binary Classification
🚀 Training model...
✅ Training completed in 3.36 minutes


In [19]:
# Evaluate Random Forest - Binary
print("\n📈 Evaluating Random Forest - Binary Classification...")

rf_binary_preds = rf_binary_model.transform(test_df)

# Binary metrics
binary_evaluator_auc = BinaryClassificationEvaluator(
    labelCol='binary_label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

binary_evaluator_pr = BinaryClassificationEvaluator(
    labelCol='binary_label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderPR'
)

multi_evaluator = MulticlassClassificationEvaluator(
    labelCol='binary_label',
    predictionCol='prediction'
)

auc_roc = binary_evaluator_auc.evaluate(rf_binary_preds)
auc_pr = binary_evaluator_pr.evaluate(rf_binary_preds)
accuracy = multi_evaluator.evaluate(rf_binary_preds, {multi_evaluator.metricName: 'accuracy'})
f1 = multi_evaluator.evaluate(rf_binary_preds, {multi_evaluator.metricName: 'f1'})
precision = multi_evaluator.evaluate(rf_binary_preds, {multi_evaluator.metricName: 'weightedPrecision'})
recall = multi_evaluator.evaluate(rf_binary_preds, {multi_evaluator.metricName: 'weightedRecall'})

print("\n" + "="*50)
print("Random Forest - Binary Classification Results")
print("="*50)
print(f"AUC-ROC:   {auc_roc:.4f}")
print(f"AUC-PR:    {auc_pr:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

rf_binary_results = {
    'model': 'Random Forest',
    'task': 'Binary Classification',
    'auc_roc': auc_roc,
    'auc_pr': auc_pr,
    'accuracy': accuracy,
    'f1': f1,
    'precision': precision,
    'recall': recall
}

# Show confusion matrix
print("\n📊 Confusion Matrix:")
rf_binary_preds.groupBy("binary_label", "prediction").count().orderBy("binary_label", "prediction").show()

rf_binary_preds.unpersist()
gc.collect()


📈 Evaluating Random Forest - Binary Classification...

Random Forest - Binary Classification Results
AUC-ROC:   0.9862
AUC-PR:    0.9933
Accuracy:  0.8968
F1 Score:  0.8995
Precision: 0.9172
Recall:    0.8968

📊 Confusion Matrix:
+------------+----------+------+
|binary_label|prediction| count|
+------------+----------+------+
|         0.0|       0.0| 54883|
|         0.0|       1.0|  1117|
|         1.0|       0.0| 16978|
|         1.0|       1.0|102363|
+------------+----------+------+



187

### 8.2 Gradient Boosted Trees - Binary Classification

In [20]:
# GBT tuning for Binary Classification
print("="*60)
print("Tuning Gradient Boosted Trees - Binary Classification")
print("="*60)

gc.collect()

gbt_param_grid = [
    {"maxIter": 80, "maxDepth": 8, "stepSize": 0.08},
    {"maxIter": 120, "maxDepth": 8, "stepSize": 0.06},
    {"maxIter": 100, "maxDepth": 10, "stepSize": 0.05},
]

best_gbt = None
best_gbt_metrics = None
best_auc = -1.0

for idx, params in enumerate(gbt_param_grid, start=1):
    print(f"\n[{idx}/{len(gbt_param_grid)}] params={params}")
    start_time = time.time()

    gbt_candidate = GBTClassifier(
        featuresCol='features_scaled',
        labelCol='binary_label',
        weightCol='binary_weight',
        maxIter=params["maxIter"],
        maxDepth=params["maxDepth"],
        stepSize=params["stepSize"],
        seed=42,
    )

    model = gbt_candidate.fit(train_df)
    preds = model.transform(test_df)

    auc = binary_evaluator_auc.evaluate(preds)
    acc = multi_evaluator.evaluate(preds, {multi_evaluator.metricName: 'accuracy'})
    f1_local = multi_evaluator.evaluate(preds, {multi_evaluator.metricName: 'f1'})

    print(f"  AUC={auc:.4f} Accuracy={acc:.4f} F1={f1_local:.4f} ({(time.time()-start_time)/60:.2f} min)")

    if auc > best_auc:
        best_auc = auc
        best_gbt = model
        best_gbt_metrics = {
            "auc_roc": auc,
            "accuracy": acc,
            "f1": f1_local,
            "params": params,
        }

    preds.unpersist()

gbt_binary_model = best_gbt
print("\n✅ Best GBT selected:")
print(best_gbt_metrics)

Tuning Gradient Boosted Trees - Binary Classification

[1/3] params={'maxIter': 80, 'maxDepth': 8, 'stepSize': 0.08}
  AUC=0.9839 Accuracy=0.8939 F1=0.8967 (1.53 min)

[2/3] params={'maxIter': 120, 'maxDepth': 8, 'stepSize': 0.06}
  AUC=0.9839 Accuracy=0.8937 F1=0.8966 (2.32 min)

[3/3] params={'maxIter': 100, 'maxDepth': 10, 'stepSize': 0.05}
  AUC=0.9811 Accuracy=0.8939 F1=0.8968 (3.34 min)

✅ Best GBT selected:
{'auc_roc': 0.983878327424894, 'accuracy': 0.8938525501736616, 'f1': 0.8967154412530374, 'params': {'maxIter': 80, 'maxDepth': 8, 'stepSize': 0.08}}


In [21]:
# Evaluate best GBT - Binary
print("\n📈 Evaluating Best GBT - Binary Classification...")

gbt_binary_preds = gbt_binary_model.transform(test_df)

auc_roc = binary_evaluator_auc.evaluate(gbt_binary_preds)
auc_pr = binary_evaluator_pr.evaluate(gbt_binary_preds)
accuracy = multi_evaluator.evaluate(gbt_binary_preds, {multi_evaluator.metricName: 'accuracy'})
f1 = multi_evaluator.evaluate(gbt_binary_preds, {multi_evaluator.metricName: 'f1'})
precision = multi_evaluator.evaluate(gbt_binary_preds, {multi_evaluator.metricName: 'weightedPrecision'})
recall = multi_evaluator.evaluate(gbt_binary_preds, {multi_evaluator.metricName: 'weightedRecall'})

print("\n" + "="*50)
print("Best GBT - Binary Classification Results")
print("="*50)
print(f"AUC-ROC:   {auc_roc:.4f}")
print(f"AUC-PR:    {auc_pr:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

gbt_binary_results = {
    'model': 'Gradient Boosted Trees',
    'task': 'Binary Classification',
    'auc_roc': auc_roc,
    'auc_pr': auc_pr,
    'accuracy': accuracy,
    'f1': f1,
    'precision': precision,
    'recall': recall,
    'best_params': best_gbt_metrics['params'] if best_gbt_metrics else None,
}

# Show confusion matrix
print("\n📊 Confusion Matrix:")
gbt_binary_preds.groupBy("binary_label", "prediction").count().orderBy("binary_label", "prediction").show()

gbt_binary_preds.unpersist()
gc.collect()


📈 Evaluating Best GBT - Binary Classification...

Best GBT - Binary Classification Results
AUC-ROC:   0.9836
AUC-PR:    0.9916
Accuracy:  0.8939
F1 Score:  0.8967
Precision: 0.9155
Recall:    0.8939

📊 Confusion Matrix:
+------------+----------+------+
|binary_label|prediction| count|
+------------+----------+------+
|         0.0|       0.0| 54893|
|         0.0|       1.0|  1107|
|         1.0|       0.0| 17505|
|         1.0|       1.0|101836|
+------------+----------+------+



1391

## Step 9: Train Multi-class Classification Model

### 9.1 Random Forest - Multi-class (10 attack categories)

In [24]:
# Attack-only Multi-class training with tuning (numeric + categorical signals)
print("="*60)
print("Tuning Random Forest - Multi-class Classification (Attack rows only)")
print("="*60)

gc.collect()

# Build dedicated multiclass datasets including categorical columns
train_mc_raw = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True)
test_mc_raw = spark.read.csv(TEST_PATH, header=True, inferSchema=True)

# Clean attack_cat and numeric columns
train_mc_raw = train_mc_raw.withColumn("attack_cat", F.trim(F.col("attack_cat")))
train_mc_raw = train_mc_raw.withColumn("attack_cat", F.when(F.col("attack_cat").isNull() | (F.col("attack_cat") == ""), "Normal").otherwise(F.col("attack_cat")))
test_mc_raw = test_mc_raw.withColumn("attack_cat", F.trim(F.col("attack_cat")))
test_mc_raw = test_mc_raw.withColumn("attack_cat", F.when(F.col("attack_cat").isNull() | (F.col("attack_cat") == ""), "Normal").otherwise(F.col("attack_cat")))

for col in NUMERIC_FEATURES:
    if col in train_mc_raw.columns:
        train_mc_raw = train_mc_raw.withColumn(
            col,
            F.when(F.col(col).isNull(), 0.0)
            .when(F.col(col) == float("inf"), 0.0)
            .when(F.col(col) == float("-inf"), 0.0)
            .otherwise(F.col(col).cast(DoubleType())),
        )
        test_mc_raw = test_mc_raw.withColumn(
            col,
            F.when(F.col(col).isNull(), 0.0)
            .when(F.col(col) == float("inf"), 0.0)
            .when(F.col(col) == float("-inf"), 0.0)
            .otherwise(F.col(col).cast(DoubleType())),
        )

# Labels
label_expr = F.lit(0)
for attack, label in ATTACK_MAPPING.items():
    label_expr = F.when(F.col("attack_cat") == attack, label).otherwise(label_expr)

train_mc_raw = train_mc_raw.withColumn("multiclass_label", label_expr.cast(DoubleType()))
test_mc_raw = test_mc_raw.withColumn("multiclass_label", label_expr.cast(DoubleType()))

# Attack-only rows
train_attack = train_mc_raw.filter(F.col("multiclass_label") > 0.0)
test_attack = test_mc_raw.filter(F.col("multiclass_label") > 0.0)

# Numeric vector + scaling
assembler_mc_num = VectorAssembler(
    inputCols=NUMERIC_FEATURES,
    outputCol="features_raw",
    handleInvalid="skip"
)
train_attack = assembler_mc_num.transform(train_attack)
test_attack = assembler_mc_num.transform(test_attack)

scaler_mc = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=True,
    withStd=True,
)
scaler_mc_model = scaler_mc.fit(train_attack)
train_attack = scaler_mc_model.transform(train_attack)
test_attack = scaler_mc_model.transform(test_attack)

# Encode categorical cols
idx_proto = StringIndexer(inputCol="proto", outputCol="proto_idx", handleInvalid="keep")
idx_service = StringIndexer(inputCol="service", outputCol="service_idx", handleInvalid="keep")
idx_state = StringIndexer(inputCol="state", outputCol="state_idx", handleInvalid="keep")

idx_proto_model = idx_proto.fit(train_attack)
train_attack = idx_proto_model.transform(train_attack)
test_attack = idx_proto_model.transform(test_attack)

idx_service_model = idx_service.fit(train_attack)
train_attack = idx_service_model.transform(train_attack)
test_attack = idx_service_model.transform(test_attack)

idx_state_model = idx_state.fit(train_attack)
train_attack = idx_state_model.transform(train_attack)
test_attack = idx_state_model.transform(test_attack)

# Final multiclass feature vector includes categorical signal
assembler_mc_final = VectorAssembler(
    inputCols=["features_scaled", "proto_idx", "service_idx", "state_idx"],
    outputCol="features_mc",
    handleInvalid="skip",
)

train_attack = assembler_mc_final.transform(train_attack)
test_attack = assembler_mc_final.transform(test_attack)

# Ensure maxBins supports categorical cardinality (required by Spark trees)
proto_bins = len(idx_proto_model.labels) + 1
service_bins = len(idx_service_model.labels) + 1
state_bins = len(idx_state_model.labels) + 1
required_max_bins = max(proto_bins, service_bins, state_bins)
print(f"Categorical bins required: proto={proto_bins}, service={service_bins}, state={state_bins}")
print(f"Using maxBins >= {required_max_bins}")

# Class weights on attack-only set
multi_counts_attack = train_attack.groupBy("multiclass_label").count().collect()
total_attack = sum(row['count'] for row in multi_counts_attack)
num_attack_classes = len(multi_counts_attack)

attack_weights = {}
for row in multi_counts_attack:
    label = row['multiclass_label']
    count = row['count']
    weight = (total_attack / (num_attack_classes * count)) ** 0.5
    attack_weights[label] = weight

attack_weight_expr = F.lit(1.0)
for label, weight in attack_weights.items():
    attack_weight_expr = F.when(F.col("multiclass_label") == label, weight).otherwise(attack_weight_expr)

train_attack = train_attack.withColumn("attack_weight", attack_weight_expr)

mc_param_grid = [
    {"numTrees": 300, "maxDepth": 18, "maxBins": 128, "minInstancesPerNode": 2},
    {"numTrees": 400, "maxDepth": 20, "maxBins": 128, "minInstancesPerNode": 2},
    {"numTrees": 500, "maxDepth": 22, "maxBins": 192, "minInstancesPerNode": 1},
]

mc_evaluator = MulticlassClassificationEvaluator(
    labelCol='multiclass_label',
    predictionCol='prediction'
)

best_mc = None
best_mc_metrics = None
best_mc_score = -1.0

for idx, params in enumerate(mc_param_grid, start=1):
    effective_max_bins = max(params['maxBins'], required_max_bins)
    print(f"\n[{idx}/{len(mc_param_grid)}] params={params}, effective_maxBins={effective_max_bins}")
    start_time = time.time()

    rf_candidate = RandomForestClassifier(
        featuresCol='features_mc',
        labelCol='multiclass_label',
        weightCol='attack_weight',
        numTrees=params['numTrees'],
        maxDepth=params['maxDepth'],
        maxBins=effective_max_bins,
        minInstancesPerNode=params['minInstancesPerNode'],
        featureSubsetStrategy='sqrt',
        seed=42,
    )

    model = rf_candidate.fit(train_attack)
    preds = model.transform(test_attack)

    acc = mc_evaluator.evaluate(preds, {mc_evaluator.metricName: 'accuracy'})
    f1_local = mc_evaluator.evaluate(preds, {mc_evaluator.metricName: 'f1'})

    print(f"  Accuracy={acc:.4f} F1={f1_local:.4f} ({(time.time()-start_time)/60:.2f} min)")

    score = (0.7 * acc) + (0.3 * f1_local)
    if score > best_mc_score:
        best_mc_score = score
        best_mc = model
        best_mc_metrics = {
            'accuracy': acc,
            'f1': f1_local,
            'params': {**params, 'maxBins': effective_max_bins},
        }

    preds.unpersist()

rf_multi_model = best_mc
print("\n✅ Best multiclass RF selected:")
print(best_mc_metrics)

# Keep for downstream evaluation cell
rf_multi_preds = rf_multi_model.transform(test_attack)
multiclass_test_rows = test_attack.count()

Tuning Random Forest - Multi-class Classification (Attack rows only)
Categorical bins required: proto=130, service=14, state=7
Using maxBins >= 130

[1/3] params={'numTrees': 300, 'maxDepth': 18, 'maxBins': 128, 'minInstancesPerNode': 2}, effective_maxBins=130
  Accuracy=0.7714 F1=0.7831 (34.86 min)

[2/3] params={'numTrees': 400, 'maxDepth': 20, 'maxBins': 128, 'minInstancesPerNode': 2}, effective_maxBins=130
  Accuracy=0.7714 F1=0.7816 (39.71 min)

[3/3] params={'numTrees': 500, 'maxDepth': 22, 'maxBins': 192, 'minInstancesPerNode': 1}, effective_maxBins=192
  Accuracy=0.7708 F1=0.7798 (51.06 min)

✅ Best multiclass RF selected:
{'accuracy': 0.7713945752088553, 'f1': 0.7830689751422633, 'params': {'numTrees': 300, 'maxDepth': 18, 'maxBins': 130, 'minInstancesPerNode': 2}}


In [25]:
# Evaluate best Random Forest - Multi-class
print("\n📈 Evaluating Best Random Forest - Multi-class Classification...")

# rf_multi_preds is produced in tuning cell
accuracy = mc_evaluator.evaluate(rf_multi_preds, {mc_evaluator.metricName: 'accuracy'})
f1 = mc_evaluator.evaluate(rf_multi_preds, {mc_evaluator.metricName: 'f1'})
precision = mc_evaluator.evaluate(rf_multi_preds, {mc_evaluator.metricName: 'weightedPrecision'})
recall = mc_evaluator.evaluate(rf_multi_preds, {mc_evaluator.metricName: 'weightedRecall'})

print("\n" + "="*50)
print("Best Random Forest - Multi-class Classification Results")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Test rows (attack-only): {multiclass_test_rows:,}")

rf_multi_results = {
    'model': 'Random Forest',
    'task': 'Multi-class Classification (attack-only)',
    'accuracy': accuracy,
    'f1': f1,
    'precision': precision,
    'recall': recall,
    'best_params': best_mc_metrics['params'] if best_mc_metrics else None,
}


📈 Evaluating Best Random Forest - Multi-class Classification...

Best Random Forest - Multi-class Classification Results
Accuracy:  0.7714
F1 Score:  0.7831
Precision: 0.8223
Recall:    0.7714
Test rows (attack-only): 119,341


In [26]:
# Per-class accuracy analysis
print("\n📊 Per-Class Performance:")

reverse_mapping = {v: k for k, v in ATTACK_MAPPING.items()}

per_class_stats = rf_multi_preds.withColumn(
    'correct', F.when(F.col('multiclass_label') == F.col('prediction'), 1).otherwise(0)
).groupBy('multiclass_label').agg(
    F.count('*').alias('total'),
    F.sum('correct').alias('correct'),
    F.round(F.sum('correct') / F.count('*'), 4).alias('accuracy')
).orderBy('multiclass_label')

per_class_stats.show()

stats = per_class_stats.collect()
print("\n" + "-"*60)
print(f"{'Class':<5} {'Attack Type':<20} {'Accuracy':>10} {'Correct/Total':>15}")
print("-"*60)
for row in stats:
    label = int(row['multiclass_label'])
    attack_name = reverse_mapping.get(label, "Unknown")
    print(f"{label:<5} {attack_name:<20} {row['accuracy']*100:>9.2f}% {row['correct']:>6}/{row['total']:<8}")

gc.collect()


📊 Per-Class Performance:


+----------------+-----+-------+--------+
|multiclass_label|total|correct|accuracy|
+----------------+-----+-------+--------+
|             1.0|18184|  15634|  0.8598|
|             2.0| 2000|     33|  0.0165|
|             3.0| 1746|    205|  0.1174|
|             4.0|12264|   8865|  0.7228|
|             5.0|33393|  19177|  0.5743|
|             6.0|40000|  39291|  0.9823|
|             7.0|10491|   7999|  0.7625|
|             8.0| 1133|    816|  0.7202|
|             9.0|  130|     39|     0.3|
+----------------+-----+-------+--------+


------------------------------------------------------------
Class Attack Type            Accuracy   Correct/Total
------------------------------------------------------------
1     Fuzzers                  85.98%  15634/18184   
2     Analysis                  1.65%     33/2000    
3     Backdoor                 11.74%    205/1746    
4     DoS                      72.28%   8865/12264   
5     Exploits                 57.43%  19177/33393   
6     

3514

## Step 10: Save Trained Models

In [27]:
# Save all models
print("💾 Saving trained models...")

# Save Random Forest - Binary
rf_binary_path = f"{MODEL_DIR}/unsw_rf_binary_classifier"
rf_binary_model.write().overwrite().save(rf_binary_path)
print(f"✅ Saved: {rf_binary_path}")

# Save GBT - Binary
gbt_binary_path = f"{MODEL_DIR}/unsw_gbt_binary_classifier"
gbt_binary_model.write().overwrite().save(gbt_binary_path)
print(f"✅ Saved: {gbt_binary_path}")

# Save Random Forest - Multi-class
rf_multi_path = f"{MODEL_DIR}/unsw_rf_multiclass_classifier"
rf_multi_model.write().overwrite().save(rf_multi_path)
print(f"✅ Saved: {rf_multi_path}")

💾 Saving trained models...


✅ Saved: D:\Coding\network-intrusion-detection\models/unsw_rf_binary_classifier
✅ Saved: D:\Coding\network-intrusion-detection\models/unsw_gbt_binary_classifier
✅ Saved: D:\Coding\network-intrusion-detection\models/unsw_rf_multiclass_classifier


In [28]:
# Save feature list and label mappings
import json

# Save feature names
feature_info = {
    'features': available_features,
    'num_features': len(available_features)
}
with open(f"{MODEL_DIR}/unsw_feature_names.json", 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"✅ Feature names saved")

# Save attack mapping
label_info = {
    'attack_mapping': ATTACK_MAPPING,
    'reverse_mapping': {str(v): k for k, v in ATTACK_MAPPING.items()}
}
with open(f"{MODEL_DIR}/unsw_label_mapping.json", 'w') as f:
    json.dump(label_info, f, indent=2)
print(f"✅ Label mapping saved")

✅ Feature names saved
✅ Label mapping saved


In [29]:
# Save training results summary
unsw_results = {
    'dataset': 'UNSW-NB15',
    'rf_binary': rf_binary_results,
    'gbt_binary': gbt_binary_results,
    'rf_multiclass': rf_multi_results,
    'train_size': train_count,
    'test_size': test_count,
    'multiclass_test_size': multiclass_test_rows,
    'num_features': len(available_features),
    'num_classes': len(ATTACK_MAPPING),
    'multiclass_strategy': 'attack_only',
}

results_path = f"{MODEL_DIR}/unsw_training_results.json"
with open(results_path, 'w') as f:
    json.dump(unsw_results, f, indent=2)

print(f"\n✅ Results saved to: {results_path}")


✅ Results saved to: D:\Coding\network-intrusion-detection\models/unsw_training_results.json


## Step 11: Model Comparison Summary

In [30]:
# Print final comparison
print("\n" + "="*70)
print("UNSW-NB15 MODEL TRAINING SUMMARY")
print("="*70)

print("\n📊 BINARY CLASSIFICATION (Attack vs Normal)")
print("-"*70)
print(f"{'Model':<25} {'AUC-ROC':<10} {'Accuracy':<10} {'F1':<10} {'Precision':<10} {'Recall':<10}")
print("-"*70)
print(f"{'Random Forest':<25} {rf_binary_results['auc_roc']:<10.4f} {rf_binary_results['accuracy']:<10.4f} {rf_binary_results['f1']:<10.4f} {rf_binary_results['precision']:<10.4f} {rf_binary_results['recall']:<10.4f}")
print(f"{'Gradient Boosted Trees':<25} {gbt_binary_results['auc_roc']:<10.4f} {gbt_binary_results['accuracy']:<10.4f} {gbt_binary_results['f1']:<10.4f} {gbt_binary_results['precision']:<10.4f} {gbt_binary_results['recall']:<10.4f}")

print("\n📊 MULTI-CLASS CLASSIFICATION (10 Attack Types)")
print("-"*70)
print(f"{'Model':<25} {'Accuracy':<10} {'F1':<10} {'Precision':<10} {'Recall':<10}")
print("-"*70)
print(f"{'Random Forest':<25} {rf_multi_results['accuracy']:<10.4f} {rf_multi_results['f1']:<10.4f} {rf_multi_results['precision']:<10.4f} {rf_multi_results['recall']:<10.4f}")

print("\n" + "="*70)
print("✅ All models trained and saved successfully!")
print(f"📁 Models location: {MODEL_DIR}")
print("="*70)


UNSW-NB15 MODEL TRAINING SUMMARY

📊 BINARY CLASSIFICATION (Attack vs Normal)
----------------------------------------------------------------------
Model                     AUC-ROC    Accuracy   F1         Precision  Recall    
----------------------------------------------------------------------
Random Forest             0.9862     0.8968     0.8995     0.9172     0.8968    
Gradient Boosted Trees    0.9836     0.8939     0.8967     0.9155     0.8939    

📊 MULTI-CLASS CLASSIFICATION (10 Attack Types)
----------------------------------------------------------------------
Model                     Accuracy   F1         Precision  Recall    
----------------------------------------------------------------------
Random Forest             0.7714     0.7831     0.8223     0.7714    

✅ All models trained and saved successfully!
📁 Models location: D:\Coding\network-intrusion-detection\models


## Step 12: Feature Importance Analysis

In [31]:
# Get feature importance from Random Forest Binary
print("📊 Top 20 Most Important Features (Random Forest - Binary):")
print("="*50)

importances = rf_binary_model.featureImportances.toArray()

# Create feature importance list
feature_importance = [(available_features[i], imp) for i, imp in enumerate(importances)]
feature_importance.sort(key=lambda x: x[1], reverse=True)

print(f"{'Rank':<6} {'Feature':<25} {'Importance':<12}")
print("-"*45)
for rank, (feat, imp) in enumerate(feature_importance[:20], 1):
    print(f"{rank:<6} {feat:<25} {imp:.6f}")

📊 Top 20 Most Important Features (Random Forest - Binary):
Rank   Feature                   Importance  
---------------------------------------------
1      sttl                      0.103710
2      ct_dst_src_ltm            0.100071
3      ct_state_ttl              0.088786
4      sbytes                    0.062535
5      smean                     0.056980
6      ct_dst_sport_ltm          0.056491
7      sload                     0.048655
8      ct_srv_dst                0.047677
9      rate                      0.047505
10     dbytes                    0.040620
11     dur                       0.030506
12     dload                     0.029476
13     tcprtt                    0.027419
14     dmean                     0.025007
15     ct_srv_src                0.024867
16     synack                    0.023014
17     dttl                      0.019091
18     sinpkt                    0.017413
19     dinpkt                    0.016601
20     sloss                     0.015480


In [34]:
# Get feature importance from Random Forest Multiclass
print("\n📊 Top 20 Most Important Features (Random Forest - Multiclass):")
print("="*50)

importances_multi = rf_multi_model.featureImportances.toArray()

# Multiclass model uses numeric features plus categorical indices
candidate_names = available_features + ["proto_idx", "service_idx", "state_idx"]

if len(candidate_names) == len(importances_multi):
    mc_feature_names = candidate_names
else:
    # Safe fallback if vector dimensions differ for any reason
    mc_feature_names = [f"feature_{i}" for i in range(len(importances_multi))]

# Create feature importance list
feature_importance_multi = [(mc_feature_names[i], imp) for i, imp in enumerate(importances_multi)]
feature_importance_multi.sort(key=lambda x: x[1], reverse=True)

print(f"{'Rank':<6} {'Feature':<25} {'Importance':<12}")
print("-"*45)
for rank, (feat, imp) in enumerate(feature_importance_multi[:20], 1):
    print(f"{rank:<6} {feat:<25} {imp:.6f}")


📊 Top 20 Most Important Features (Random Forest - Multiclass):
Rank   Feature                   Importance  
---------------------------------------------
1      sbytes                    0.142643
2      service_idx               0.113000
3      smean                     0.102900
4      ct_srv_dst                0.061428
5      proto_idx                 0.059893
6      ct_srv_src                0.046098
7      dbytes                    0.036317
8      ct_src_dport_ltm          0.034927
9      sttl                      0.031117
10     dmean                     0.029406
11     ct_dst_src_ltm            0.028841
12     ct_dst_sport_ltm          0.026511
13     sload                     0.023776
14     ct_dst_ltm                0.021026
15     dur                       0.019384
16     sinpkt                    0.016065
17     sjit                      0.015987
18     dloss                     0.014946
19     spkts                     0.014930
20     ct_src_ltm                0.014323


## Summary

### Models Trained:
1. **Random Forest - Binary** (`unsw_rf_binary_classifier`)
   - Task: Attack vs Normal detection
   - Use case: Quick attack detection

2. **Gradient Boosted Trees - Binary** (`unsw_gbt_binary_classifier`)
   - Task: Attack vs Normal detection
   - Use case: Higher accuracy attack detection

3. **Random Forest - Multi-class** (`unsw_rf_multiclass_classifier`)
   - Task: Identify specific attack type (10 classes)
   - Use case: Detailed threat classification

### Saved Artifacts:
- Models: `models/unsw_*`
- Scaler: `models/unsw_scaler`
- Feature names: `models/unsw_feature_names.json`
- Label mapping: `models/unsw_label_mapping.json`
- Results: `models/unsw_training_results.json`

### Next Steps:
1. Deploy models for real-time inference
2. Integrate with Kafka streaming pipeline
3. Build alerting/monitoring dashboard

In [35]:
# Cleanup
print("Cleaning up...")
try:
    train_df.unpersist()
    test_df.unpersist()
except:
    pass

gc.collect()
spark.stop()
print("✅ Spark session stopped")
print("\n🎉 UNSW-NB15 model training complete! Ready for deployment.")

Cleaning up...
✅ Spark session stopped

🎉 UNSW-NB15 model training complete! Ready for deployment.
